**Cell 1**

# NB14 — Llama-2-7B HelpSteer2 DPO experts, seed 137

**Purpose.** Does a larger base model produce more specialised experts and a different
relationship matrix `R`? NB14 repeats the NB11 DPO recipe on `meta-llama/Llama-2-7b-hf`
with one training seed. The TinyLlama counterpart is the NB11 seed-137 run used in NB13
(`nb11_pairs2690_seed137_run1`): same seed, same 2,690 pairs per attribute.

**Identical to NB11:** HelpSteer2 revision, pair rule, pair seed 137, 2,690 pairs per
attribute (pair logic imported unchanged from `scripts/0_dpo_expert.py`), LoRA r=8, alpha=16,
dropout 0.05 on all seven projection modules, DPO beta 0.1, sigmoid loss, 1 epoch,
lr 5e-4 cosine with 3 % warmup, effective batch 8, max length 512 / prompt 256, bf16,
fixed final checkpoint, ArmoRM never loaded.

**Different from NB11, and why:**

| | NB11 (TinyLlama) | NB14 (Llama-2-7B) | reason |
|---|---|---|---|
| base | TinyLlama-1.1B-Chat | Llama-2-7b-hf (raw) | research question; same base as the assistant task |
| prompt format | TinyLlama chat template | `\n\nHuman: {prompt} \n\nAssistant:` | raw base has no chat template; supervisor's assistant-task format |
| tokenizer | fast | slow (`use_fast=False`) | as in the supervisor's `load_main_tokenizer` |
| batch shape | 2 × 4 | 1 × 8 | same effective batch, fits a 40 GB A100 |
| checkpoints | local, final copy to Drive | written directly to Drive every 20 steps | ~3–5 h per attribute must survive a Colab disconnect |

**Script:** `scripts/0_dpo_expert_llama2.py` (new, additive; `0_dpo_expert.py` is untouched).

**Requirements.** A100 GPU. A Hugging Face token with access to `meta-llama/Llama-2-7b-hf`
(accept the licence on the model page), stored as Colab secret `HF_TOKEN`.

**Runtime (estimate).** About 3–5 h per attribute, 15–25 h in total. Run all is supported and
resumable: after a disconnect, simply run all again. Finished attributes are verified and
skipped, an interrupted one resumes from its last checkpoint on Drive.

**Outputs.** Five adapters on Drive; geometry (`R_cos`, `R_gram`, norms, floor LP for 75
preferences, comparison with TinyLlama seed 137) in the repository results folder.


**Cell 2**

Mount Drive, then clone or update the repository.


In [ ]:
# Cell 3
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis
assert os.path.isfile("scripts/0_dpo_expert_llama2.py"), "Push scripts/0_dpo_expert_llama2.py to the repository first."


**Cell 4**

GPU check. bf16 is required; the batch shape is chosen for 40 GB.


In [ ]:
# Cell 5
!nvidia-smi
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select an A100 runtime."
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)}, {VRAM_GIB:.1f} GiB")
assert VRAM_GIB >= 38, "Llama-2-7B DPO needs an A100 (40 or 80 GB)."


**Cell 6**

Pinned NB11 DPO stack (identical to NB11.1 Cell 7). `USE_TF=0` avoids the Colab TensorFlow/protobuf conflict seen in NB13.2.


In [ ]:
# Cell 7
import importlib.metadata, subprocess, sys
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
print(f"Python {sys.version.split()[0]} ({sys.executable})")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "protobuf==5.29.5", "transformers==4.45.2",
    "tokenizers==0.20.3", "peft==0.13.2", "accelerate==1.1.1", "trl==0.11.4",
    "datasets==3.1.0", "huggingface_hub==0.36.0", "bitsandbytes", "pyyaml", "safetensors",
    "sentencepiece", "scipy==1.14.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages], check=True)
expected = {"transformers": "4.45.2", "trl": "0.11.4"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, f"Incompatible DPO stack: {installed}. Restart the runtime and run all again."
print({n: importlib.metadata.version(n) for n in ("transformers", "tokenizers", "peft", "accelerate", "trl", "datasets")})


**Cell 8**

Hugging Face access and the frozen base revision. The revision is resolved once and stored on Drive, so every attribute and every later rerun uses the same weights.


In [ ]:
# Cell 9
from pathlib import Path
import json
from google.colab import userdata
from huggingface_hub import HfApi, login

BASE_MODEL = "meta-llama/Llama-2-7b-hf"
RUN_TAG = "nb14_llama2_7b_helpsteer2_dpo_seed137"
DRIVE_ROOT = Path("/content/drive/MyDrive/master-thesis-nb14") / RUN_TAG
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add your Hugging Face token as Colab secret HF_TOKEN."
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

revision_file = DRIVE_ROOT / "base_revision.json"
if revision_file.is_file():
    BASE_REVISION = json.loads(revision_file.read_text())["base_revision"]
else:
    BASE_REVISION = HfApi().model_info(BASE_MODEL).sha
    revision_file.write_text(json.dumps({"base_model": BASE_MODEL, "base_revision": BASE_REVISION}, indent=2))
assert BASE_REVISION and len(BASE_REVISION) == 40
print(f"{BASE_MODEL} @ {BASE_REVISION}")


**Cell 10**

Frozen settings. Everything except base, prompt format, tokenizer and batch shape is copied from NB11.1.


In [ ]:
# Cell 11
PROJECT_ROOT = Path("/content/master-thesis").resolve()
DATASET_NAME = "nvidia/HelpSteer2"
DATASET_REVISION = "990b2711a36180dd19d9c94b8627844866f8982a"
ATTRIBUTES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]
SEED = 137
PAIR_SEED = 137
MAX_PAIRS = 2690
DPO_BETA = 0.1
EPOCHS = 1.0
LEARNING_RATE = 5e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
SAVE_STEPS = 20
RESPONSE_SPLIT = "\n\nAssistant:"      # Instructions.response_split of the supervisor's utils.py
RUN_SMOKE_TEST = True
DOWNLOAD_ADAPTER_BUNDLE = False        # about 0.4 GB; the bundle is always copied to Drive

TRAINED_ROOT = DRIVE_ROOT / "trained" / f"seed_{SEED}"   # training writes here directly
RESULTS_DIR = PROJECT_ROOT / "results" / RUN_TAG / "geometry"
TRAINED_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# The supervisor's utils.py, if it is in the repository, must agree on the response split.
utils_path = PROJECT_ROOT / "baselines/utils/utils.py"
if utils_path.is_file():
    sys.path.insert(0, str(PROJECT_ROOT))
    from baselines.utils.utils import Instructions
    assert Instructions.response_split == RESPONSE_SPLIT, Instructions.response_split
    print("[OK] response split matches baselines/utils/utils.py")
else:
    print("[note] baselines/utils/utils.py not in the repository; using the default response split")
print(f"training root (Drive) = {TRAINED_ROOT}\nresults (repo)        = {RESULTS_DIR}")


**Cell 12**

Fix the pair subsets. Pair construction is the unchanged NB11 code, so the hashes must equal those of the NB11 seed-137 run for the same attribute.


In [ ]:
# Cell 13
import hashlib, importlib.util
from datasets import load_dataset

def load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / "scripts" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

llama_script = load_script("nb14_dpo_script", "0_dpo_expert_llama2.py")
nb11_script = llama_script.nb11

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

train_rows = load_dataset(DATASET_NAME, split="train", revision=DATASET_REVISION)
PAIR_ID_HASHES = {}
for axis in ATTRIBUTES:
    selected = nb11_script.select_pairs(nb11_script.build_pairs_from_rows(train_rows, axis),
                                        seed=PAIR_SEED, max_pairs=MAX_PAIRS)
    PAIR_ID_HASHES[axis] = nb11_script.canonical_hash({"pair_ids": [p["pair_id"] for p in selected]})
    print(f"{axis:12} selected={len(selected)} pairs_sha256={PAIR_ID_HASHES[axis][:16]}...")
example = llama_script.format_pairs_hh(selected[:1], type("T", (), {"eos_token": "</s>"})(), RESPONSE_SPLIT)[0]
print("\nExample training prompt:", repr(example["prompt"][:160]))
del train_rows


**Cell 14**

Verification of a finished expert against the frozen settings.


In [ ]:
# Cell 15
EXPECTED_LORA = {"r": 8, "alpha": 16, "dropout": 0.05, "bias": "none",
                 "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}

def expert_dir(axis):
    return TRAINED_ROOT / f"dpo_{axis}"

def verify_expert(axis):
    directory = expert_dir(axis)
    manifest_path = directory / "training_manifest.json"
    weights_path = directory / "adapter/adapter_model.safetensors"
    if not manifest_path.is_file() or not weights_path.is_file():
        raise FileNotFoundError(f"Incomplete expert: {directory}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {
        "completed": True, "training_method": "DPO", "reward_name": axis,
        "base_model_name": BASE_MODEL, "base_revision": BASE_REVISION,
        "dataset_name": DATASET_NAME, "dataset_revision": DATASET_REVISION, "dataset_split": "train",
        "pair_seed": PAIR_SEED, "selected_pair_count": MAX_PAIRS,
        "selected_pair_ids_sha256": PAIR_ID_HASHES[axis], "response_split": RESPONSE_SPLIT,
        "beta": DPO_BETA, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM, "max_length": MAX_LENGTH,
        "max_prompt_length": MAX_PROMPT_LENGTH, "precision": "bf16", "seed": SEED,
        "armorm_used_during_training": False,
        "checkpoint_selection": "fixed final epoch; no reward-model selection",
    }
    for field, value in expected.items():
        if manifest.get(field) != value:
            raise RuntimeError(f"{directory} differs on {field}: {manifest.get(field)!r} != {value!r}")
    if manifest.get("lora") != EXPECTED_LORA:
        raise RuntimeError(f"{directory} has a different LoRA configuration")
    if sha256_file(weights_path) != manifest.get("adapter_model_sha256"):
        raise RuntimeError(f"Adapter hash mismatch: {directory}")
    return manifest

def is_complete(axis):
    try:
        verify_expert(axis)
        return True
    except FileNotFoundError:
        return False

print({axis: is_complete(axis) for axis in ATTRIBUTES})


**Cell 16**

Smoke test with 16 pairs (two optimizer steps). It runs only while at least one attribute is missing and catches loading, memory and API problems before the long run.


In [ ]:
# Cell 17
from collections import deque
import time

def run_logged_subprocess(command, label):
    print(f"[run] {label}", flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(command, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
    if process.wait():
        raise RuntimeError(f"{label} failed.\nLast output:\n" + "\n".join(tail))

def script_command(axis, output_root, max_pairs, resume=True, overwrite=False):
    command = [sys.executable, "-u", "scripts/0_dpo_expert_llama2.py",
               "--reward_name", axis, "--base_model_name", BASE_MODEL, "--base_revision", BASE_REVISION,
               "--dataset_name", DATASET_NAME, "--dataset_revision", DATASET_REVISION, "--split", "train",
               "--output_root", str(output_root), "--response_split", RESPONSE_SPLIT,
               "--beta", str(DPO_BETA), "--epochs", str(EPOCHS), "--lr", str(LEARNING_RATE),
               "--batch_size", str(BATCH_SIZE), "--grad_accum", str(GRAD_ACCUM),
               "--max_length", str(MAX_LENGTH), "--max_prompt_length", str(MAX_PROMPT_LENGTH),
               "--max_pairs", str(max_pairs), "--pair_seed", str(PAIR_SEED), "--seed", str(SEED),
               "--save_steps", str(SAVE_STEPS)]
    if resume:
        command.append("--resume")
    if overwrite:
        command.append("--overwrite")
    return command

if RUN_SMOKE_TEST and not all(is_complete(axis) for axis in ATTRIBUTES):
    smoke_root = Path("/tmp/nb14_dpo_smoke")
    shutil.rmtree(smoke_root, ignore_errors=True)
    t0 = time.perf_counter()
    run_logged_subprocess(script_command("helpfulness", smoke_root, 16, resume=False, overwrite=True), "DPO smoke test")
    print(f"[OK] smoke test passed in {time.perf_counter() - t0:.0f} s (includes model download on first run)")
    print(f"peak GPU memory reported by the subprocess is not visible here; watch nvidia-smi in Cell 5 if needed")
    shutil.rmtree(smoke_root, ignore_errors=True)
else:
    print("[skip] smoke test")


**Cell 18**

Training of all missing attributes. This is the long-running cell. Re-running it is safe: finished attributes are verified and skipped, an interrupted attribute resumes from its last checkpoint on Drive. Wall-clock time per attribute is recorded.


In [ ]:
# Cell 19
import gc
TIMES_PATH = DRIVE_ROOT / "training_wall_clock.json"
times = json.loads(TIMES_PATH.read_text()) if TIMES_PATH.is_file() else {}

for number, axis in enumerate(ATTRIBUTES, start=1):
    print(f"\n##### {number}/{len(ATTRIBUTES)}: {axis} #####")
    if is_complete(axis):
        print(f"[skip] verified {axis}")
        continue
    t0 = time.perf_counter()
    run_logged_subprocess(script_command(axis, TRAINED_ROOT, MAX_PAIRS), f"DPO {axis}")
    verify_expert(axis)
    times[axis] = time.perf_counter() - t0   # last completed segment; an interrupted earlier segment is not counted
    TIMES_PATH.write_text(json.dumps(times, indent=2))
    trainer_dir = expert_dir(axis) / "trainer"
    if trainer_dir.exists():
        shutil.rmtree(trainer_dir)          # final adapter is verified on Drive; checkpoints no longer needed
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[OK] {axis}: {times[axis] / 3600:.2f} h")
print("[OK] all five Llama-2-7B DPO experts are complete.")


**Cell 20**

Inventory of the five experts.


In [ ]:
# Cell 21
import pandas as pd
inventory = []
for axis in ATTRIBUTES:
    manifest = verify_expert(axis)
    metrics = manifest.get("train_metrics", {})
    inventory.append({
        "attribute": axis, "adapter_model_sha256": manifest["adapter_model_sha256"],
        "selected_pair_ids_sha256": manifest["selected_pair_ids_sha256"],
        "train_runtime_s_last_segment": metrics.get("train_runtime"),
        "train_loss": metrics.get("train_loss"),
        "wall_clock_h_total": times.get(axis, float("nan")) / 3600 if axis in times else None,
    })
INVENTORY_DF = pd.DataFrame(inventory)
INVENTORY_DF.to_csv(RESULTS_DIR / "adapter_inventory.csv", index=False)
display(INVENTORY_DF)


**Cell 22**

Geometry: effective-update Gram and cosine matrices, norms, eigenvalues, `R⁻¹·1`, and the p-aware floor LP for the same 75 preferences as NB13.1 (11 named + 64 Dirichlet, seed 137). Primary matrix is `R_cos`, as in NB13.


In [ ]:
# Cell 23
import numpy as np
sys.path.insert(0, str(PROJECT_ROOT))
from src.effective_lora_geometry import (effective_lora_inner_product, effective_lora_update_norm,
    effective_lora_update_numel, load_effective_lora_geometry, validate_compatible_geometries)
from src.coefficient_portfolio import floor_lp_at_p
from src.experiment_config import load_experiment_config, validate_preference_vectors
from src.proxy_validation import build_search_set

geometries = {axis: load_effective_lora_geometry(expert_dir(axis) / "adapter") for axis in ATTRIBUTES}
validate_compatible_geometries([geometries[a] for a in ATTRIBUTES], ATTRIBUTES)
m = len(ATTRIBUTES)
R_gram = np.empty((m, m))
for i, left in enumerate(ATTRIBUTES):
    for j, right in enumerate(ATTRIBUTES[:i + 1]):
        R_gram[i, j] = R_gram[j, i] = effective_lora_inner_product(geometries[left], geometries[right])
norms = np.sqrt(np.diag(R_gram))
for i, axis in enumerate(ATTRIBUTES):
    assert np.isclose(norms[i], effective_lora_update_norm(geometries[axis]), rtol=1e-10)
R_cos = 0.5 * (R_gram / np.outer(norms, norms) + (R_gram / np.outer(norms, norms)).T)
eig = np.linalg.eigvalsh(R_cos)
offdiag = R_cos[np.triu_indices(m, 1)]
inv_ones = np.linalg.solve(R_cos, np.ones(m))

config = load_experiment_config(PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml")
PREFERENCES = validate_preference_vectors(config)
pref_set = [(name, np.asarray(v, float)) for name, v in PREFERENCES.items()]
seen = {tuple(np.round(v, 9)) for _, v in pref_set}
for index, vector in enumerate(build_search_set(m, n_dirichlet=64, dirichlet_alpha=1.0,
                                                preferences=list(PREFERENCES.values()), seed=137)):
    if tuple(np.round(vector, 9)) not in seen:
        seen.add(tuple(np.round(vector, 9)))
        pref_set.append((f"dirichlet_{index:02d}", np.asarray(vector, float)))
floor_rows = [{"p_name": name, "t_lp": float(t), "collapsed": bool(c)}
              for name, p in pref_set for t, c in [floor_lp_at_p(R_cos, p)]]
FLOOR_DF = pd.DataFrame(floor_rows)

pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(RESULTS_DIR / "R_llama2_dpo_cos.csv")
pd.DataFrame(R_gram, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(RESULTS_DIR / "R_llama2_dpo_gram.csv")
pd.DataFrame({"attribute": ATTRIBUTES, "effective_update_norm": norms}).to_csv(RESULTS_DIR / "llama2_dpo_update_norms.csv", index=False)
FLOOR_DF.to_csv(RESULTS_DIR / "floor_lp.csv", index=False)
GEOMETRY = {
    "regime": "helpsteer2_dpo_llama2_7b_seed137", "primary_matrix": "R_cos",
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION, "attribute_order": ATTRIBUTES,
    "effective_update_numel": int(effective_lora_update_numel(geometries[ATTRIBUTES[0]])),
    "update_norms": dict(zip(ATTRIBUTES, norms.tolist())),
    "cosine_eigenvalues": eig.tolist(),
    "cosine_condition_number": float(eig.max() / eig.min()),
    "cosine_dominant_eigenvalue_share": float(eig.max() / eig.sum()),
    "cosine_offdiag_min": float(offdiag.min()), "cosine_offdiag_max": float(offdiag.max()),
    "cosine_offdiag_mean": float(offdiag.mean()),
    "inverse_R_ones": inv_ones.tolist(), "inverse_R_ones_strictly_positive": bool(np.all(inv_ones > 0)),
    "floor_lp": {"n_preferences": len(FLOOR_DF), "n_collapsed": int(FLOOR_DF["collapsed"].sum())},
}
(RESULTS_DIR / "llama2_dpo_geometry_report.json").write_text(json.dumps(GEOMETRY, indent=2) + "\n")
display(pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).round(4))
print(json.dumps({k: v for k, v in GEOMETRY.items() if k not in ("cosine_eigenvalues", "update_norms")}, indent=2))


**Cell 24**

Comparison with the TinyLlama counterpart (NB11 seed 137, as used in NB13).


In [ ]:
# Cell 25
tiny_dir = PROJECT_ROOT / "results/nb13_nb11_dpo_run1_outputs"
R_tiny = pd.read_csv(tiny_dir / "R_dpo_cos.csv", index_col=0).loc[ATTRIBUTES, ATTRIBUTES].to_numpy()
tiny_norms = pd.read_csv(tiny_dir / "dpo_update_norms.csv").set_index("attribute").loc[ATTRIBUTES, "effective_update_norm"].to_numpy()

def summary(R):
    e = np.linalg.eigvalsh(R); off = R[np.triu_indices(m, 1)]
    return {"offdiag_min": off.min(), "offdiag_max": off.max(), "offdiag_mean": off.mean(),
            "n_negative_offdiag": int((off < 0).sum()), "dominant_share": e.max() / e.sum(),
            "condition_number": e.max() / e.min(),
            "floor_collapses_(R^-1 1 > 0)": bool(np.all(np.linalg.solve(R, np.ones(m)) > 0))}

COMPARISON = pd.DataFrame({"TinyLlama-1.1B (NB11 seed 137)": summary(R_tiny),
                           "Llama-2-7B (NB14 seed 137)": summary(R_cos)})
COMPARISON.loc["update_norm_mean"] = [tiny_norms.mean(), norms.mean()]
COMPARISON.to_csv(RESULTS_DIR / "comparison_tinyllama_vs_llama2.csv")
display(COMPARISON)
print("\nElementwise difference R_llama2 - R_tinyllama:")
display(pd.DataFrame(R_cos - R_tiny, index=ATTRIBUTES, columns=ATTRIBUTES).round(4))


**Cell 26**

Export. The small geometry folder goes into the repository results; the five adapters are bundled on Drive for later evaluation notebooks.


In [ ]:
# Cell 27
import zipfile
from datetime import datetime, timezone
from google.colab import files

RUN_MANIFEST = {
    "schema_version": 1, "notebook": "NB14 Llama-2-7B HelpSteer2 DPO experts, seed 137",
    "created_utc": datetime.now(timezone.utc).isoformat(), "run_tag": RUN_TAG,
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
    "dataset": DATASET_NAME, "dataset_revision": DATASET_REVISION,
    "seed": SEED, "pair_seed": PAIR_SEED, "n_pairs_per_attribute": MAX_PAIRS,
    "prompt_format": "\\n\\nHuman: {prompt} " + RESPONSE_SPLIT.replace("\n", "\\n"),
    "training_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert_llama2.py"),
    "pair_logic_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert.py"),
    "adapter_hashes": {row["attribute"]: row["adapter_model_sha256"] for row in inventory},
    "pair_hashes": PAIR_ID_HASHES,
    "training_wall_clock_h": {k: v / 3600 for k, v in times.items()},
    "tinyllama_counterpart": "results/dpo_rq2/nb11_pairs2690_seed137_run1 (NB13 adapters)",
    "armorm_used_during_training": False,
}
(RESULTS_DIR / "nb14_run_manifest.json").write_text(json.dumps(RUN_MANIFEST, indent=2) + "\n")

geometry_zip = Path(f"/content/{RUN_TAG}_geometry.zip")
with zipfile.ZipFile(geometry_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.iterdir()):
        archive.write(path, f"results/{RUN_TAG}/geometry/{path.name}")
shutil.copy2(geometry_zip, DRIVE_ROOT / geometry_zip.name)

adapter_zip = DRIVE_ROOT / f"{RUN_TAG}_adapters.zip"
with zipfile.ZipFile(adapter_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for axis in ATTRIBUTES:
        for path in sorted((expert_dir(axis)).rglob("*")):
            if path.is_file() and "trainer" not in path.parts:
                archive.write(path, f"dpo_{axis}/{path.relative_to(expert_dir(axis)).as_posix()}")
    archive.write(RESULTS_DIR / "nb14_run_manifest.json", "nb14_run_manifest.json")
print(f"geometry ZIP : {geometry_zip}\nadapter ZIP  : {adapter_zip} ({adapter_zip.stat().st_size / 1e6:.0f} MB, SHA256 {sha256_file(adapter_zip)})")
files.download(str(geometry_zip))
if DOWNLOAD_ADAPTER_BUNDLE:
    files.download(str(adapter_zip))


**Cell 28**

Repository status.


In [ ]:
# Cell 29
!git status --short
